# Pipelines de treinamento: XGBoost

Esse notebook contém as pipelines de treinamento usadas na obtenção do melhor modelo classificador do problema desenvolvido no EP2.

O modelo LogisticRegression aceita diversas parametrizações diferentes - veja documentação: https://xgboost.readthedocs.io/en/stable/python/index.html.

As **features** que exploramos são:  

1. Bag of Words  
2. TF/TF-IDF  
3. CHAR NGrams  
4. Embeddings

Os **modelos de embeddings** que exploramos aqui são:  

1. BAAI bge-3


Você pode encontrar uma execução já parametrizada do melhor modelo encontrado por essas pipelines no notebook model.ipynb, **que é nossa versão de entrega do EP2**.

## Bootstrap Imports

In [1]:
import pandas as pd
import numpy as np

import os
import sys
from pathlib import Path

### Teste de importação: Lib.utils do projeto

In [2]:
filedir = Path(os.getcwd())
base_path = filedir.resolve().parents[3]
sys.path.append(str(base_path))

from Lib.utils import printhello
printhello()

HELLO!


## Configura variáveis de execução

In [3]:
sep = ";"
dec = ","
quotech = "\""
encoding = "latin-1"


EP_dir = "EP2"
CSV_input_name = "ep2-train.csv"
path_to_archive = f"../../../../Traindata/{EP_dir}/{CSV_input_name}"


do_print = True
if do_print:
    print(f"Path to csv input is:  {path_to_archive}")

Path to csv input is:  ../../../../Traindata/EP2/ep2-train.csv


### Configure variáveis de reprodutibilidade

In [4]:
random_state = 12345

### Lista de melhores modelos

In [5]:
best_models_list = []

## Pré-tratamento de dados

### Importar dados do csv

In [6]:
df = pd.read_csv(path_to_archive, na_values=['na'],
sep=sep,
decimal=dec,
quotechar=quotech,
encoding=encoding,
encoding_errors='strict')
print(df.shape)
print(df.columns)

(43678, 2)
Index(['req_text', 'profession'], dtype='object')


### Embaralhamento dos dados

O .csv de entrada tem alto ordenamento dos inputs por classe. Carregá-los dessa maneira nos modelos p/ treinamento introduz viés, então é preciso embaralhar os dados para garantir randomicidade. 
As classes em sklearn.model_selection - como a StratfiedKFold usada mais a frente - implementam parâmetro shuffle="", que pode ser passado como True para embaralhar mais os dados.

Note que é importante também garantir a reprodutibilidade do embaralhamento, especificando um valor hardcoded (Neste caso random_state=100)

In [7]:
print("Shape antes do shuffle:", df.shape)

df = df.sample(frac=1, random_state=random_state).reset_index(drop=True) #NAO MUDE random_state, essa variavel DEVE valer 12345, ou QUEBRARÁ REPRODUTIBILIDADE dos experimentos

print("Shape depois do shuffle:", df.shape)

Shape antes do shuffle: (43678, 2)
Shape depois do shuffle: (43678, 2)


### Limpeza dos dados

In [8]:
from Lib.utils import clean_text
#def clean_text(text, do_lowercase: bool, rem_emails: bool, rem_urls: bool, normalize_whitespaces: bool):

df['req_text_cleaned'] = df['req_text'].apply(lambda row_text: clean_text(
        row_text, 
        do_lowercase=True, 
        rem_emails=True, 
        rem_urls=True, 
        normalize_whitespaces=True
    ))

df['req_text'] = df['req_text_cleaned']
df = df.drop(columns=['req_text_cleaned']) # Remove a coluna temporária

# Treino dos modelos XGBoost

## Imports

In [9]:
import xgboost as xgb
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from imblearn.pipeline  import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import SelectKBest, chi2
from imblearn.under_sampling import RandomUnderSampler

## Training Features: Bag of Words, TF-IDF, Word NGram

### Disable Warnings

In [10]:
import warnings
from sklearn.exceptions import ConvergenceWarning

# Ignora warnings de ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Ignora UserWarnings específicos de l1_ratio etc
warnings.filterwarnings("ignore", category=UserWarning)

### Feature: Bag of words 

#### Definição da Pipeline

In [11]:
BoW_pipeline = Pipeline([
    ('vect', CountVectorizer()),
    ('undersample', RandomUnderSampler(random_state=random_state)),
    ('kbest', SelectKBest(score_func=chi2)),
    ('classifier', xgb.XGBClassifier(early_stopping_rounds=3)), 
])  

### Undersampling

In [ ]:
undersampler = RandomUnderSampler(
    sampling_strategy={
        'government': 10303,
        'academic': 10303,
        'private': 10303,
    },
    random_state=random_state
)
normalizer = Normalizer(norm='l2') 

X_under, y_under = undersampler.fit_resample(df["req_text"].fillna(""), df["profession"].values) #Gera X e y undersampleados

#### Treinamento

In [12]:
BoW_parameters = {
    'undersample__sampling_strategy': [
        {
            'government': 10303,
            'academic': 10303,
            'private': 10303,
        },
        {
            'government': 18782,
            'academic': 14593,
            'private': 10303,
        }],
    'kbest__k': [50, 100, 200, 500, 750, 1500, 4000, 10000],
    'classifier__objective': ['multi:softmax'], #softprob seria interessante, mas nao usaremos
    'classifier__use_label_encoder': [False],
    'classifier__eval_metric': ['logloss'],
    'classifier__n_estimators': [50, 75, 100, 150, 200, 250],
    'classifier__learning_rate': [0.02, 0.1, 0.2],
    'classifier__max_depth': [3, 5, 7],
    'classifier__colsample_bytree': [0.8, 1.0],
}

In [13]:
BoW_classifier = GridSearchCV(BoW_pipeline, BoW_parameters, 
                                       cv=10, n_jobs=-1, scoring="accuracy", verbose=1, error_score = np.nan)
BoW_classifier.fit(df["req_text"].fillna(""), df["profession"].values)

Fitting 10 folds for each of 1728 candidates, totalling 17280 fits


ValueError: 
All the 17280 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
6048 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/imblearn/pipeline.py", line 518, in fit
    Xt, yt = self._fit(X, y, routed_params, raw_params=params)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/imblearn/pipeline.py", li

In [ ]:
print("Melhor acurácia média:", BoW_classifier.best_score_)
print("Melhores parâmetros:", BoW_classifier.best_params_)

best_models_list.append({
    "features": "BoW",
    "accuracy": BoW_classifier.best_score_,
    "params": BoW_classifier.best_params_,
})

### Feature: TF-IDF

#### Definição da Pipeline

In [ ]:
TF_pipeline = Pipeline([
    ('undersample', RandomUnderSampler(random_state=random_state)),
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('kbest', SelectKBest(score_func=chi2)),
    ('classifier', xgb.XGBClassifier(early_stopping_rounds=3)), 
])

#### Treinamento

In [ ]:
TF_parameters = {
    'undersample__sampling_strategy': [
        {
            'government': 10303,
            'academic': 10303,
            'private': 10303,
        },
        {
            'government': 18782,
            'academic': 14593,
            'private': 10303,
        }],
    'tfidf__use_idf': [True, False],
    'kbest__k': [50, 100, 200, 500, 750, 1500, 4000, 10000],
    'classifier__objective': ['multi:softmax'], #softprob seria interessante, mas nao
    'classifier__use_label_encoder': [False],
    'classifier__eval_metric': ['logloss'],
    'classifier__n_estimators': [50, 75, 100, 150, 200, 250],
    'classifier__learning_rate': [0.02, 0.1, 0.2],
    'classifier__max_depth': [3, 5, 7],
    'classifier__colsample_bytree': [0.8, 1.0],
}

In [ ]:
TF_classifier = GridSearchCV(TF_pipeline, TF_parameters, 
                                       cv=10, n_jobs=-1, scoring="accuracy", verbose=1, error_score = np.nan)
TF_classifier.fit(df["req_text"].fillna(""), df["profession"].values)

In [ ]:
print("Melhor acurácia média:", TF_classifier.best_score_)
print("Melhores parâmetros:", TF_classifier.best_params_)

best_models_list.append({
    "features": "TF",
    "accuracy": TF_classifier.best_score_,
    "params": TF_classifier.best_params_,
})

### Feature: CHAR Ngram

#### Definição da Pipeline

In [ ]:
NGram_pipeline = Pipeline([
    ('undersample', RandomUnderSampler(random_state=random_state)),
    ('vect', CountVectorizer()),
    ('kbest', SelectKBest(score_func=chi2)),
    ('classifier', xgb.XGBClassifier(early_stopping_rounds=3)), 
]) 

#### Treinamento

In [ ]:
CHAR_NGram_parameters = { 
    'undersample__sampling_strategy': [
        {
            'government': 10303,
            'academic': 10303,
            'private': 10303,
        },
        {
            'government': 18782,
            'academic': 14593,
            'private': 10303,
        }],
    'vect__ngram_range': [(1, 1), (2, 2), (3, 3), (4, 4), (5, 5), (6, 6), (7, 7), (8, 8), (9, 9), (10, 10), (11, 11), (12, 12)],
    'vect__analyzer': ["char", "char_wb"],
    'kbest__k': [100, 200, 750, 4000, 10000],
    'classifier__objective': ['multi:softmax'], #softprob seria interessante, mas nao
    'classifier__use_label_encoder': [False],
    'classifier__eval_metric': ['logloss'],
    'classifier__n_estimators': [50, 75, 100, 150, 200, 250],
    'classifier__learning_rate': [0.02, 0.1, 0.2],
    'classifier__max_depth': [3, 5, 7],
    'classifier__colsample_bytree': [0.8, 1.0],
}

In [ ]:
CHAR_NGram_classifier = GridSearchCV(NGram_pipeline, CHAR_NGram_parameters, 
                                       cv=10, n_jobs=2, scoring="accuracy", verbose=1, error_score = np.nan)
CHAR_NGram_classifier.fit(df["req_text"].fillna(""), df["profession"].values)

In [ ]:
print("Melhor acurácia média:", CHAR_NGram_classifier.best_score_)
print("Melhores parâmetros:", CHAR_NGram_classifier.best_params_)

best_models_list.append({
    "features": "CHAR-NGram",
    "accuracy": CHAR_NGram_classifier.best_score_,
    "params": CHAR_NGram_classifier.best_params_,
})

## Training Features: Embeddings

### Algoritmo & parâmetros

In [ ]:
xgb_pipeline_embeddings = Pipeline([
    ('undersample', RandomUnderSampler(random_state=random_state)),
    ('classifier', xgb.XGBClassifier(early_stopping_rounds=3)), 
]) 

xgb_parameters = {
    'undersample__sampling_strategy': [
        {
            'government': 10303,
            'academic': 10303,
            'private': 10303,
        },
        {
            'government': 18782,
            'academic': 14593,
            'private': 10303,
        }],
    'classifier__objective': ['multi:softmax'], #softprob seria interessante, mas nao
    'classifier__use_label_encoder': [False],
    'classifier__eval_metric': ['logloss'],
    'classifier__n_estimators': [50, 75, 100, 150, 200, 250],
    'classifier__learning_rate': [0.02, 0.1, 0.2],
    'classifier__max_depth': [3, 5, 7],
    'classifier__colsample_bytree': [0.8, 1.0],
}


### Treino c/ embeddings do modelo: BAAI-bge-3

#### Importação dos embeddings

In [ ]:
# 1. Gerar os dados X e Y
X_baai = np.load('../../Embeddings/npys/gen_baai_bge3.ipynb')
y_baai = df["profession"].values

Gerando BAAI embeddings
Shape dos embeddings (X): (500, 1024)
embeddings gerados


#### Treino do modelo

In [ ]:
xgb_grid_baai = GridSearchCV(xgb_pipeline_embeddings, xgb_parameters, 
                            cv=10, n_jobs=-1, scoring="accuracy", verbose=1, error_score=np.nan)
xgb_grid_baai.fit(X_baai, y_baai)

In [ ]:
print("Melhor acurácia média:", xgb_grid_baai.best_score_)
print("Melhores parâmetros:", xgb_grid_baai.best_params_)

best_models_list.append({
    "feature": "embeddings baai-bge-3",
    "accuracy": xgb_grid_baai.best_score_,
    "params": xgb_grid_baai.best_params_,
})

# Seleciona melhores parâmetros

In [ ]:
best_score = -1
best = 0
for idx, candidate in enumerate(best_models_list):
    if candidate["accuracy"] > best_score:
        best = idx
        best_score = candidate["accuracy"]

print(f"O melhor classificador encontrado pelas pipelines é -->    feature={best_models_list[best]["features"]}\n")
print(f"Melhor acucácia encontrada:  {best_models_list[best]['accuracy']}")
print(f"Melhores parametros encontrados:  {best_models_list[best]['params']}")